# 模块三：全国 31 省多源异构数据“终极大缝合” (Grand Merge)

**执行人**：成员 4 (数据工程大师)

**合并策略与难点攻克**：
1. **多 Sheet 精准制导**：摒弃手动拆表，利用 `pandas` 的 `sheet_name` 参数，直接从队友提交的多 Sheet Excel 中抽取出名为“省级面板数据”的核心表。
2. **脏列名洗脱 (Column Stripping)**：队友提交的表头中存在大量 `\n` 换行符与不可见空格（如 `预期寿命\n(岁)`），本代码构建了自动化清洗逻辑，将其统一展平。
3. **主键强对齐 (Key Alignment)**：将各表的 `Province`、`Year` 强制映射为 `地区` 和 `年份`，并再次调用正则表达式切除行政区划后缀，确保与底座 100% 匹配。
4. **非平衡面板处理**：考虑到健康产出端（成员3）仅含 2010、2020-2022 年数据，环境端（成员2）PM2.5 仅含 2013-2024 年数据，合并采用 `Outer Join` 最大化保留信息，并采用双轨制输出策略。

In [2]:
import pandas as pd
import re
import warnings
warnings.filterwarnings('ignore')

print("========== 🚀 终极大缝合引擎启动 ==========\n")

# 1. 载入我们打好的“地基”
base_path = r'D:\2026_BigData_Project\data\processed\China_Province_Health_Panel.csv'
df_base = pd.read_csv(base_path)

# 2. 构建多源数据精准读取配置字典
file_configs = {
    "投入端_成员1": {
        "path": r'D:\2026_BigData_Project\data\raw\成员1-硬核医疗资源与财政投入.xlsx',
        "sheet": "省级面板数据"
    },
    "环境端_成员2": {
        "path": r'D:\2026_BigData_Project\data\raw\成员2-各省社会经济与环境本底.xlsx',
        "sheet": "省级面板数据"
    },
    "产出端_成员3": {
        "path": r'D:\2026_BigData_Project\data\raw\成员3-各省人口结构与健康终局结果.xlsx',
        "sheet": "省级面板数据"
    }
}

# 核心脱水函数：切除省份后缀
def clean_province(series):
    return series.astype(str).str.replace(r'(维吾尔自治区|壮族自治区|回族自治区|自治区|省|市)$', '', regex=True).str.strip()

# 批量读取、重命名、清洗
dfs_to_merge = []
for name, config in file_configs.items():
    print(f"正在抽取 [{name}] -> Sheet: '{config['sheet']}'")
    
    # 精准读取
    df_temp = pd.read_excel(config['path'], sheet_name=config['sheet'])
    
    # 【高能排雷】清洗列名里的换行符 \n 和多余空格 (如 "预期寿命\n(岁)" -> "预期寿命(岁)")
    df_temp.columns = [str(col).replace('\n', '').replace('\r', '').strip() for col in df_temp.columns]
    
    # 统一主键名称
    rename_dict = {'Province': '地区', 'Year': '年份', 'province': '地区', 'year': '年份'}
    df_temp = df_temp.rename(columns=rename_dict)
    
    # 强制脱水与类型转换
    if '地区' in df_temp.columns:
        df_temp['地区'] = clean_province(df_temp['地区'])
    if '年份' in df_temp.columns:
        df_temp['年份'] = df_temp['年份'].astype(int)
        
    dfs_to_merge.append(df_temp)

print("\n✅ 三路外部数据清洗完毕，列名换行符已抹平，准备接入主干！")

========== 🚀 终极大缝合引擎启动 ==========

正在抽取 [投入端_成员1] -> Sheet: '省级面板数据'
正在抽取 [环境端_成员2] -> Sheet: '省级面板数据'
正在抽取 [产出端_成员3] -> Sheet: '省级面板数据'

✅ 三路外部数据清洗完毕，列名换行符已抹平，准备接入主干！


In [3]:
print("========== 🔗 执行多表级联 Merge ==========\n")

# 1. 以 df_base 为主干，依次执行 Outer Join（外连接），保底所有数据不丢失
df_final = df_base.copy()

for i, df_ext in enumerate(dfs_to_merge):
    # 找出除了[地区, 年份]之外的指标列，防止把队友表里多余的空列也并进来
    cols_to_use = ['地区', '年份'] + [col for col in df_ext.columns if col not in ['地区', '年份']]
    
    df_final = pd.merge(df_final, df_ext[cols_to_use], on=['地区', '年份'], how='outer')

# 按照地区和年份排个序
df_final = df_final.sort_values(by=['地区', '年份']).reset_index(drop=True)

print(f"✅ 终极缝合完毕！大宽表共计 {df_final.shape[0]} 行，{df_final.shape[1]} 列。")
print(f"包含的指标字段：{list(df_final.columns)[2:8]} ... (等 {df_final.shape[1]-2} 个核心指标)")

# 2. 导出【全量画图表】(给成员 5 画图用，保留所有 NaN)
out_full_path = r'D:\2026_BigData_Project\data\processed\Final_Master_Dataset_Full.csv'
df_final.to_csv(out_full_path, index=False)
print(f"\n📁 [轨道一] 全量数据集已导出至：{out_full_path}")
print("   -> 请将此表发给【成员5】，供其绘制近20年医疗资源增长地图等全局图表。")

========== 🔗 执行多表级联 Merge ==========

✅ 终极缝合完毕！大宽表共计 775 行，33 列。
包含的指标字段：['卫生人员数量(万人)', '医疗机构数量(个)', '卫生技术人员总数(人)', '医疗卫生机构总数(个)', '每千人口床位数(张/千人)', '医疗卫生一般公共预算支出(万元)'] ... (等 31 个核心指标)

📁 [轨道一] 全量数据集已导出至：D:\2026_BigData_Project\data\processed\Final_Master_Dataset_Full.csv
   -> 请将此表发给【成员5】，供其绘制近20年医疗资源增长地图等全局图表。


In [4]:
print("========== 🎓 启动特殊挂载：教育本底数据 ==========\n")

# 1. 单独读取成员2的“教育”Sheet
edu_path = r'D:\2026_BigData_Project\data\raw\成员2-各省社会经济与环境本底.xlsx'
df_edu = pd.read_excel(edu_path, sheet_name='教育')

# 2. 基础清洗与脱水
df_edu.columns = [str(col).replace('\n', '').strip() for col in df_edu.columns]
rename_edu = {'Province': '地区', 'Year': '年份', 'province': '地区', 'year': '年份'}
df_edu = df_edu.rename(columns=rename_edu)

if '地区' in df_edu.columns:
    df_edu['地区'] = clean_province(df_edu['地区'])
if '年份' in df_edu.columns:
    df_edu['年份'] = df_edu['年份'].astype(int)

# 3. 提取核心指标：只拿“大学（大专及以上）”来代表高等教育水平
# 假设列名就是这个，如果报错请对照 Excel 里的真实列名修改
edu_cols = ['地区', '年份', '大学（大专及以上）']
# 防止 Excel 里有别的杂乱列，强行筛选
df_edu_core = df_edu[[col for col in edu_cols if col in df_edu.columns]]

# 4. 🧠 学术级缺失值填补 (按省份分组，进行前后填充)
# 将教育表按照年份补齐到完整的 2005-2024 年轴上
years_full = pd.DataFrame({'年份': range(2005, 2025)})
provinces = pd.DataFrame({'地区': df_base['地区'].unique()})
# 生成 31省 x 20年的完整骨架
full_skeleton = provinces.merge(years_full, how='cross')

# 把残缺的教育数据合并到骨架上
df_edu_full = pd.merge(full_skeleton, df_edu_core, on=['地区', '年份'], how='left')

# 核心魔法：按省份分组，用前一年的数据填补后一年，如果前面没有就用后一年的填补
print("正在执行受教育水平的学术级插值填补...")
df_edu_full['大学（大专及以上）'] = df_edu_full.groupby('地区')['大学（大专及以上）'].transform(lambda x: x.ffill().bfill())

# 5. 最终并入主干！
# 把填补好的教育数据 Merge 进我们刚才的 df_final 里
df_final = pd.merge(df_final, df_edu_full, on=['地区', '年份'], how='left')

print("✅ 教育数据成功挂载并完成插值！没有产生新的断层。")
print("挂载后的前两行预览：")
print(df_final[['地区', '年份', '大学（大专及以上）']].head(2))

========== 🎓 启动特殊挂载：教育本底数据 ==========

正在执行受教育水平的学术级插值填补...
✅ 教育数据成功挂载并完成插值！没有产生新的断层。
挂载后的前两行预览：
   地区    年份  大学（大专及以上）
0  上海  2000        NaN
1  上海  2001        NaN


In [5]:
print("\n========== 🛠️ 模型数据截断与纯净提炼 (防误杀升级版) ==========\n")

# 1. 时间截断：提取 2020-2022 这三年的黄金面板
df_strict = df_final[df_final['年份'].isin([2020, 2021, 2022])].copy()

# 2. 🕵️ 侦探环节：看看刚才到底是谁引发了“团灭”
print("🚨 【空值侦探报告】在 2020-2022 的 93 行数据中，以下列存在缺失值：")
nan_stats = df_strict.isnull().sum()
print(nan_stats[nan_stats > 0])

# 3. 💣 智能排雷：先清理“毒列”，再清理行
# 规则 A：干掉队友留下的“备注”、“数据来源”或者 Excel 自动生成的“Unnamed”幽灵列
garbage_cols = [col for col in df_strict.columns if '备注' in col or 'Unnamed' in col or '来源' in col]
if garbage_cols:
    print(f"\n🗑️ 发现并自动清剿垃圾列：{garbage_cols}")
    df_strict = df_strict.drop(columns=garbage_cols)

# 规则 B：如果有某列指标在这 3 年里大面积缺失（缺失超过一半），为了保全行数据，直接把这列踢出群聊
threshold = len(df_strict) * 0.5 
df_strict = df_strict.dropna(axis=1, thresh=threshold)

# 4. 真正安全的行级清洗
initial_rows = df_strict.shape[0]
df_strict = df_strict.dropna()
dropped_rows = initial_rows - df_strict.shape[0]

# 5. 重新导出严谨建模表
out_strict_path = r'D:\2026_BigData_Project\data\processed\Final_Model_Dataset_Strict.csv'
df_strict.to_csv(out_strict_path, index=False)

print(f"\n📁 [轨道二] 严谨建模数据集已重新导出至：{out_strict_path}")
print(f"   🧹 清洗报告：清除了 {dropped_rows} 行不完整数据。")
print(f"   🎯 最终投喂给 DEA 模型的高质量纯净样本数：{df_strict.shape[0]} 行。")

if df_strict.shape[0] == 93:
    print("   🏆 完美！31个省份3年的数据一滴没漏，全部安全保留！老六可以开干了！")
else:
    print("   ⚠️ 提示：仍然有少量行被删。请往上翻看【空值侦探报告】，看看是哪个硬核指标出了幺蛾子。")


========== 🛠️ 模型数据截断与纯净提炼 (防误杀升级版) ==========

🚨 【空值侦探报告】在 2020-2022 的 93 行数据中，以下列存在缺失值：
备注    93
dtype: int64

🗑️ 发现并自动清剿垃圾列：['备注']

📁 [轨道二] 严谨建模数据集已重新导出至：D:\2026_BigData_Project\data\processed\Final_Model_Dataset_Strict.csv
   🧹 清洗报告：清除了 0 行不完整数据。
   🎯 最终投喂给 DEA 模型的高质量纯净样本数：93 行。
   🏆 完美！31个省份3年的数据一滴没漏，全部安全保留！老六可以开干了！


In [6]:
import pandas as pd
import numpy as np

print("========== 🏥 终极数据集深度体检 (Data Quality Audit) ==========\n")

# 1. 载入刚刚生成的两份终极神表
strict_path = r'D:\2026_BigData_Project\data\processed\Final_Model_Dataset_Strict.csv'
full_path = r'D:\2026_BigData_Project\data\processed\Final_Master_Dataset_Full.csv'

df_strict = pd.read_csv(strict_path)
df_full = pd.read_csv(full_path)

# ==========================================
# 【体检项 1】：严谨建模表 (给老六的纯净数据)
# ==========================================
print("【一】严谨建模表 (Final_Model_Dataset_Strict) 体检报告：")
print(f"👉 预期：93行 (31省*3年)，0缺失。")
print(f"📊 实际尺寸：{df_strict.shape[0]} 行, {df_strict.shape[1]} 列")

# 1.1 扫描致命空值
nan_strict = df_strict.isnull().sum().sum()
if nan_strict == 0:
    print("🧩 缺失值扫描：✅ 完美！缺失值为 0。")
else:
    print(f"❌ 警告：发现 {nan_strict} 个残余空值！")

# 1.2 扫描逻辑异常值 (是否有违背常理的负数)
numeric_cols = df_strict.select_dtypes(include=[np.number]).columns
negative_counts = (df_strict[numeric_cols] < 0).sum()
if negative_counts.sum() == 0:
    print("📈 异常值扫描：✅ 完美！没有任何指标出现“负数”这种反常识错误。")
else:
    print("⚠️ 警告：发现异常负数！请核查以下列：")
    print(negative_counts[negative_counts > 0])

print("\n--------------------------------------------------\n")

# ==========================================
# 【体检项 2】：全量画图表 (给老五的 20 年大表)
# ==========================================
print("【二】全量画图表 (Final_Master_Dataset_Full) 体检报告：")
print(f"👉 预期：620行 (31省*20年)。")
print(f"📊 实际尺寸：{df_full.shape[0]} 行, {df_full.shape[1]} 列")

# 2.1 扫描省份与年份完整度 (防止合并时弄丢了省份)
provinces_count = df_full['地区'].nunique()
years_count = df_full['年份'].nunique()
years_min = df_full['年份'].min()
years_max = df_full['年份'].max()

if provinces_count == 31:
    print(f"🗺️ 省份覆盖：✅ 完美！整整齐齐 {provinces_count} 个省份。")
else:
    print(f"❌ 警告：省份数量不对！当前为 {provinces_count} 个。")

if years_count >= 20 and years_min <= 2005 and years_max >= 2024:
    print(f"📅 年份覆盖：✅ 完美！时间轴 {years_min}-{years_max} 连续未断裂。")
else:
    print(f"⚠️ 警告：时间轴出现异常，当前覆盖：{years_min} 到 {years_max}。")

# ==========================================
# 【体检项 3】：极端极值雷达 (看一眼有没有夸张的数据)
# ==========================================
print("\n【三】极值雷达侦测 (抽查核心指标最大值)：")
try:
    print(f"   - 最高老龄化率: {df_strict['老龄化率(65岁以上占比)(%)'].max()}%")
    print(f"   - 最高预期寿命: {df_strict['预期寿命(岁)'].max()} 岁")
    print(f"   - 最高人均医疗支出: {df_strict['人均医疗卫生财政支出(元)'].max()} 元")
    print("   (👆 请人工扫一眼这些数字，如果预期寿命出现 200 岁，说明队友数据找错了)")
except KeyError:
    print("   (跳过极值雷达：部分列名可能存在差异)")

print("\n==================================================")
if nan_strict == 0 and provinces_count == 31 and negative_counts.sum() == 0:
    print("🎉 最终诊断：超级健康！双表逻辑自洽，防弹级别！大胆发给队友吧！")

========== 🏥 终极数据集深度体检 (Data Quality Audit) ==========

【一】严谨建模表 (Final_Model_Dataset_Strict) 体检报告：
👉 预期：93行 (31省*3年)，0缺失。
📊 实际尺寸：93 行, 33 列
🧩 缺失值扫描：✅ 完美！缺失值为 0。
📈 异常值扫描：✅ 完美！没有任何指标出现“负数”这种反常识错误。

--------------------------------------------------

【二】全量画图表 (Final_Master_Dataset_Full) 体检报告：
👉 预期：620行 (31省*20年)。
📊 实际尺寸：775 行, 33 列
🗺️ 省份覆盖：✅ 完美！整整齐齐 31 个省份。
📅 年份覆盖：✅ 完美！时间轴 2000-2024 连续未断裂。

【三】极值雷达侦测 (抽查核心指标最大值)：
   (跳过极值雷达：部分列名可能存在差异)

🎉 最终诊断：超级健康！双表逻辑自洽，防弹级别！大胆发给队友吧！


In [7]:
print("\n========== 🗺️ 模块四：最新年份医疗资源极化追踪 ==========\n")

# 1. 找到全量数据里的最新年份 (根据你的体检报告，应该是2024)
latest_year = df_full['年份'].max()
df_latest = df_full[df_full['年份'] == latest_year].copy()

# 2. 检查列名是不是准确的，如果报错请改成你们 Excel 里的真实名字
col_bed = '每千人口床位数(张/千人)'

if col_bed in df_latest.columns:
    # 去除空值后排序
    df_latest_clean = df_latest.dropna(subset=[col_bed])
    
    top5 = df_latest_clean.sort_values(by=col_bed, ascending=False).head(5)
    bottom5 = df_latest_clean.sort_values(by=col_bed, ascending=True).head(5)

    print(f"【{latest_year}年 Top 5 资源富集区】:")
    print(top5[['地区', col_bed]].to_string(index=False))

    print(f"\n【{latest_year}年 Bottom 5 效能洼地区】:")
    print(bottom5[['地区', col_bed]].to_string(index=False))
else:
    print(f"❌ 找不到列名：'{col_bed}'，请打印 df_full.columns 看一下具体的床位指标叫什么名字！")
print("==========================================================")


========== 🗺️ 模块四：最新年份医疗资源极化追踪 ==========

【2024年 Top 5 资源富集区】:
 地区  每千人口床位数(张/千人)
 贵州          11.27
 辽宁          11.16
黑龙江          11.14
 海南          11.05
 云南          10.59

【2024年 Bottom 5 效能洼地区】:
地区  每千人口床位数(张/千人)
四川           5.86
陕西           5.95
天津           6.28
湖北           6.29
河南           6.32
